# Aerial OBB Object Detection & Benchmark Suite
### Google Colab GPU Runner with Google Drive Persistence & Crash Resumption

Run this notebook in Google Colab with a **GPU runtime**:
`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`

This runner provides:
1. **Google Drive Integration**: Mounts `/content/drive/MyDrive/object-detection` for persistent storage of datasets (`data/`), model weights (`weights/`), and outputs (`results/`).
2. **Crash-Resilient Checkpointing**: Automatically checks for existing checkpoints upon restart and resumes without repeating completed work.
3. **Proactive RAM Protection**: Actively monitors memory usage, throttling batch sizes and running garbage collection to prevent Colab OOM crashes.
4. **Complete Evaluation Pipeline**: Evaluates models (`yolov8n-obb`, `custom-obb`) across datasets (`CODrone`, `VisDrone`, `DOTA`) with comprehensive metrics and visual heatmaps.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Mount Google Drive to persist all datasets, weights, and benchmark results
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# 3. Initialize Google Drive directory structure for object-detection
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/object-detection")
DRIVE_DATA = DRIVE_ROOT / "data"
DRIVE_WEIGHTS = DRIVE_ROOT / "weights"
DRIVE_RESULTS = DRIVE_ROOT / "results"

for folder in [DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_RESULTS]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"[✓] Google Drive structure initialized at: {DRIVE_ROOT}")
print(f"    - Datasets Dir : {DRIVE_DATA}")
print(f"    - Weights Dir  : {DRIVE_WEIGHTS}")
print(f"    - Results Dir  : {DRIVE_RESULTS}")

In [ ]:
# 4. Clone or pull the repository
import os
REPO_URL = "https://github.com/Addy0312/minor-project.git"
if not os.path.exists("/content/minor-project"):
    !git clone $REPO_URL /content/minor-project
%cd /content/minor-project
!git pull

In [ ]:
# 5. Install required packages
!python install.py

In [ ]:
# 6. Fast CPU/GPU Smoke Test (verifies pipeline, synthetic data, metrics, and plots in seconds)
!python scripts/run_pipeline.py --test --save-plots

In [ ]:
# 7. Run Full Benchmark on CODrone & VisDrone
# Results and checkpoints automatically persist to /content/drive/MyDrive/object-detection/results/
# If disconnected or restarted, re-running this cell automatically resumes from checkpoints!
!python scripts/run_pipeline.py \
    --datasets codrone visdrone \
    --models yolov8n-obb custom-obb \
    --device cuda \
    --save-plots \
    --resume True

In [ ]:
# 8. Display Generated Benchmark Report & Plots Inline
import glob
from pathlib import Path
from IPython.display import Image, display, Markdown

# Search in Google Drive results first, then local fallback
search_dirs = ["/content/drive/MyDrive/object-detection/results/run_*", "results/run_*"]
run_dirs = []
for pattern in search_dirs:
    run_dirs.extend(glob.glob(pattern))

run_dirs = sorted(run_dirs)
if run_dirs:
    latest_run = run_dirs[-1]
    print(f"Latest Run Directory: {latest_run}")
    report_file = os.path.join(latest_run, "benchmark_report.md")
    if os.path.exists(report_file):
        with open(report_file) as f:
            display(Markdown(f.read()))

    # Display plots
    for img_path in sorted(glob.glob(f"{latest_run}/plots/*.png")):
        print(f"Displaying: {os.path.basename(img_path)}")
        display(Image(filename=img_path))
else:
    print("No results directory found yet. Run step 7 above first.")